# Evaluation — Faithfulness + Expansion + Marketing Vibe

Đánh giá **bất kỳ file JSON nào** đã sinh ở bước generation.

**Input:** file JSON từ `generation/generate_*.ipynb` — format `[{instruction, title, seed, output}, ...]`  
**Output:**
- `results/runs/MODEL_TIMESTAMP.csv` — điểm chi tiết từng case
- `results/fnb_dataset_test.eval_summary.csv` — tổng hợp tất cả runs

---
**Workflow:**
1. Chạy notebook sinh bài tương ứng: `generation/generate_baseline.ipynb`, `generate_ollama.ipynb`, hoặc `generate_oracle.ipynb`
2. Copy đường dẫn file JSON output vào `INPUT_JSON` bên dưới
3. Đặt `MODEL_ID`
4. Chạy toàn bộ notebook này

In [43]:
# %pip install -q deepeval openai pandas python-dotenv

In [44]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
load_dotenv(_dotenv_path, override=True, encoding="utf-8")
ROOT = Path(_dotenv_path).resolve().parent if _dotenv_path else Path.cwd().resolve()

# ===== ĐỔI 2 DÒNG NÀY CHO MỖI RUN =====
INPUT_JSON = ROOT / "results/generations/qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json"
MODEL_ID   = ""              # để trống → tự lấy từ tên file
# ========================================

SUMMARY_REL        = "results/fnb_dataset_test.eval_summary.csv"
RUNS_REL           = "results/evaluations"
EVAL_PROGRESS_CHUNK = 10

def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()

JUDGE_MODEL             = (_env("JUDGE_MODEL") or _env("OPENAI_MODEL") or "gpt-4o-mini").strip()
JUDGE_AZURE_ENDPOINT    = (_env("JUDGE_AZURE_ENDPOINT") or _env("AZURE_OPENAI_ENDPOINT")).rstrip("/")
JUDGE_AZURE_API_KEY     = _env("JUDGE_AZURE_API_KEY") or _env("AZURE_OPENAI_API_KEY")
JUDGE_AZURE_API_VERSION = _env("JUDGE_AZURE_API_VERSION") or _env("OPENAI_API_VERSION") or "2024-08-01-preview"

if not (JUDGE_AZURE_ENDPOINT and JUDGE_AZURE_API_KEY):
    raise RuntimeError("Judge cần JUDGE_AZURE_ENDPOINT + JUDGE_AZURE_API_KEY (hoặc AZURE_OPENAI_*).")

os.environ["USE_AZURE_OPENAI"] = "true"
os.environ.setdefault("OPENAI_API_VERSION", JUDGE_AZURE_API_VERSION)
os.environ.pop("USE_OPENAI_MODEL", None)
os.environ["OPENAI_MODEL"] = "azure"

JUDGE_BACKEND = "azure"
SUMMARY_PATH  = ROOT / SUMMARY_REL
RUNS_DIR      = ROOT / RUNS_REL

if not INPUT_JSON.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {INPUT_JSON}")

# Auto-detect MODEL_ID từ tên file nếu không đặt thủ công
if not MODEL_ID:
    MODEL_ID = INPUT_JSON.stem

print("Input JSON:", INPUT_JSON)
print("Model ID:", MODEL_ID)
print("Judge:", JUDGE_MODEL, "@", JUDGE_AZURE_ENDPOINT)

Input JSON: D:\Github\mcs-train-content-model\results\generations\qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json
Model ID: qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05
Judge: gpt-5.4 @ https://vqnhan-poc.openai.azure.com


In [45]:
import json
from typing import List, Dict

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

if not isinstance(raw, list):
    raise ValueError("Generation JSON phải là danh sách [{id, instruction, title, seed, output, ...}]")

cases: List[Dict] = []
for idx, row in enumerate(raw):
    output = str(row.get("output", "")).strip()
    if not output:
        print(f"  ⚠ row {idx}: output rỗng — sẽ bỏ qua khi eval.")
    cases.append({
        "id":               row.get("id", idx + 1),
        "instruction":      str(row.get("instruction", "")),
        "title":            str(row.get("title", "")),
        "seed":             str(row.get("seed", "")),
        "output":           output,
        "time_generation":  row.get("time_generation"),
        "tokens_input":     row.get("tokens_input"),
        "tokens_output":    row.get("tokens_output"),
        "tokens_reasoning": row.get("tokens_reasoning"),
        "tokens_total":     row.get("tokens_total"),
    })

valid_count = sum(1 for c in cases if c["output"])
print(f"Loaded {len(cases)} cases ({valid_count} có output) từ {INPUT_JSON.name}")

Loaded 100 cases (100 có output) từ qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05.json


In [46]:
from openai import AzureOpenAI

_judge = AzureOpenAI(
    azure_endpoint=JUDGE_AZURE_ENDPOINT,
    api_key=JUDGE_AZURE_API_KEY,
    api_version=JUDGE_AZURE_API_VERSION,
)
_judge_model_id = JUDGE_MODEL

_r = _judge.chat.completions.create(
    model=_judge_model_id,
    messages=[{"role": "user", "content": "Xin chào"}],
)
print("Judge OK:", (_r.choices[0].message.content or "").strip()[:80])

Judge OK: Xin chào! Tôi có thể giúp gì cho bạn hôm nay?


In [47]:
import json
import re
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from dotenv import find_dotenv, load_dotenv

from metrics import JudgeLLM, FaithfulnessEvaluator, ExpansionQualityEvaluator, MarketingVibeEvaluator

load_dotenv(find_dotenv(usecwd=True), override=True, encoding="utf-8")

judge_llm = JudgeLLM(_judge, _judge_model_id)
print(f"JudgeLLM: {judge_llm.get_model_name()}")

RUN_ID = datetime.now(timezone.utc).strftime("%d-%m-%Y_%H-%M")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

out_json_path = RUNS_DIR / f"{INPUT_JSON.stem}_eval.json"

# Load existing results để hỗ trợ resume khi crash
results: list = []
evaluated_ids: set = set()
if out_json_path.is_file():
    with open(out_json_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    evaluated_ids = {r["id"] for r in results}
    print(f"Resume: đã có {len(results)} cases được đánh giá, bỏ qua ids: {sorted(evaluated_ids)}")

eval_cases = [c for c in cases if c["output"] and c["id"] not in evaluated_ids]
print(f"Đang đánh giá {len(eval_cases)}/{len(cases)} cases...")

if eval_cases:
    f_ev = FaithfulnessEvaluator(judge_llm)
    e_ev = ExpansionQualityEvaluator(judge_llm)
    m_ev = MarketingVibeEvaluator(judge_llm)
    t0 = time.perf_counter()

    for i, c in enumerate(eval_cases, start=1):
        fr = f_ev.evaluate_one(c["title"], c["seed"], c["output"])
        er = e_ev.evaluate_one(c["title"], c["seed"], c["output"])
        mr = m_ev.evaluate_one(c["title"], c["seed"], c["output"])

        results.append({
            "id":                       c["id"],
            "instruction":              c["instruction"],
            "title":                    c["title"],
            "seed":                     c["seed"],
            "output":                   c["output"],
            "time_generation":          c["time_generation"],
            "tokens_input":             c["tokens_input"],
            "tokens_output":            c["tokens_output"],
            "tokens_reasoning":         c["tokens_reasoning"],
            "tokens_total":             c["tokens_total"],
            "faithfulness_rule":        fr.rule_entity_score,
            "faithfulness_llm":         fr.llm_faithfulness_score,
            "faithfulness_combined":    fr.combined_score,
            "expansion_llm":            er.llm_expansion_score,
            "expansion_rule_length":    er.rule_length_score,
            "expansion_combined":       er.combined_score,
            "vibe_llm_hook":            mr.llm_hook_tone_score,
            "vibe_rule_md_cta":         mr.rule_combined,
            "vibe_combined":            mr.combined_score,
            "faithfulness_llm_reason":  (fr.llm_reason or "")[:500],
            "expansion_llm_reason":     (er.llm_reason or "")[:500],
            "vibe_llm_reason":          (mr.llm_reason or "")[:500],
        })
        results.sort(key=lambda x: x["id"])

        # Ghi ngay sau mỗi case để tránh mất dữ liệu khi crash
        with open(out_json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        if i % EVAL_PROGRESS_CHUNK == 0 or i == len(eval_cases):
            elapsed = time.perf_counter() - t0
            print(f"  {i}/{len(eval_cases)} cases — {elapsed:.1f}s", flush=True)

print(f"Đã lưu: {out_json_path} ({len(results)} cases total)")

c:\Users\vqnhan\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

JudgeLLM: gpt-5.4
Đang đánh giá 100/100 cases...


  10/100 cases — 117.5s


  20/100 cases — 226.8s


  30/100 cases — 332.0s


...

  60/100 cases — 637.0s


  70/100 cases — 733.8s


  80/100 cases — 836.3s


  90/100 cases — 936.9s


  100/100 cases — 1037.2s
Đã lưu: D:\Github\mcs-train-content-model\results\evaluations\qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05_eval.json (100 cases total)


In [48]:
import numpy as np
import pandas as pd
from datetime import datetime, timezone

summary_cols = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
scored = [r for r in results if "faithfulness_combined" in r]

if not scored:
    print("Không có case nào có kết quả đánh giá.")
else:
    means = {k: sum(r[k] for r in scored) / len(scored) for k in summary_cols}
    overall = float(np.mean(list(means.values())))

    summary_row = {
        "run_id":                 RUN_ID,
        "judge_backend":          JUDGE_BACKEND,
        "judge_model":            JUDGE_MODEL,
        "local_model_id":         MODEL_ID,
        "input_json":             str(INPUT_JSON.relative_to(ROOT)),
        "n_cases":                len(scored),
        "eval_json":              str(out_json_path.relative_to(ROOT)),
        "faithfulness_combined":  means["faithfulness_combined"],
        "expansion_combined":     means["expansion_combined"],
        "vibe_combined":          means["vibe_combined"],
        "overall_mean":           overall,
        "last_updated_utc":       datetime.now(timezone.utc).isoformat(),
    }

    new_df = pd.DataFrame([summary_row])
    if SUMMARY_PATH.is_file():
        prev_df     = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
        summary_out = pd.concat([prev_df, new_df], ignore_index=True)
    else:
        SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
        summary_out = new_df

    summary_out.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
    print(f"Đã cập nhật tổng hợp: {SUMMARY_PATH}")
    pd.DataFrame([summary_row]).T

Đã cập nhật tổng hợp: D:\Github\mcs-train-content-model\results\fnb_dataset_test.eval_summary.csv


In [49]:
# So sánh tất cả runs trong eval_summary.csv
df_all = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
cols = ["local_model_id", "run_id",
        "faithfulness_combined", "expansion_combined", "vibe_combined", "overall_mean"]
display_cols = [c for c in cols if c in df_all.columns]
df_all[display_cols].sort_values(["local_model_id", "run_id"])

,local_model_id,run_id,faithfulness_combined,expansion_combined,vibe_combined,overall_mean
0,md-gpt-5.4-mini_21-05-2026_13-53,21-05-2026_10-19,0.679448,0.778490,0.647697,0.701878
1,md-gpt-5.4-mini_21-05-2026_13-53,21-05-2026_10-39,0.679448,0.778490,0.647697,0.701878
2,md-gpt-5.4-mini_21-05-2026_14-31,21-05-2026_10-40,0.658775,0.816998,0.645071,0.706948
9,qwen3.5-2b-facebook-content_nothinking_21-05-2...,21-05-2026_15-01,0.607523,0.714035,0.593747,0.638435
10,qwen3.5-2b-facebook-content_nothinking_21-05-2...,21-05-2026_15-20,0.596982,0.754244,0.586618,0.645948
13,qwen3.5-4b-facebook-content_nothinking_21-05-2...,21-05-2026_16-36,0.632903,0.766250,0.651252,0.683469
14,qwen3.5-4b-facebook-content_nothinking_22-05-2...,21-05-2026_23-53,0.636644,0.776555,0.648197,0.687132
3,qwen3.5_2b_nothinking_21-05-2026_13-35,21-05-2026_12-56,0.586864,0.662819,0.433150,0.560944
4,qwen3.5_2b_nothinking_21-05-2026_14-01,21-05-2026_13-17,0.598781,0.661708,0.449936,0.570142
5,qwen3.5_2b_nothinking_21-05-2026_14-01,21-05-2026_13-36,0.598781,0.661708,0.449936,0.570142
